# 02 Bronze - Banking

**Audience:** participants learning the AIDP medallion pattern with PySpark.

**Prerequisites:** use the lab's shared compute and run the previous notebook first.

**Learning goal:** Preserves source values and lineage while converting each dataset to Delta.

## Outline

1. Inspect the participant-scoped inputs.
2. Transform and persist this medallion layer.
3. Register external tables when this layer owns them.
4. Verify the row counts printed by the final statements.


In [ ]:
import re
# oidlUtils is injected by AIDP Workbench; it is not an importable module.

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name)
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")

if re.fullmatch(r"u_[0-9a-f]{16}", participant_key) is None:
    raise ValueError("Invalid participant_key")
if lab_id != 'banking':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")

from pyspark.sql import functions as F

specs = {'branches': {'filename': 'branches.csv', 'primary_key': ['branch_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'branch_id', 'type': 'STRING', 'required': True}, {'name': 'branch_type', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'opened_date', 'type': 'DATE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'customers': {'filename': 'customers.csv', 'primary_key': ['customer_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'customer_id', 'type': 'STRING', 'required': True}, {'name': 'customer_type', 'type': 'STRING', 'required': True}, {'name': 'segment', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'risk_band', 'type': 'STRING', 'required': True}, {'name': 'onboarding_date', 'type': 'DATE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'accounts': {'filename': 'accounts.csv', 'primary_key': ['account_id'], 'foreign_keys': [['customer_id', 'customers', 'customer_id'], ['branch_id', 'branches', 'branch_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'account_id', 'type': 'STRING', 'required': True}, {'name': 'customer_id', 'type': 'STRING', 'required': True}, {'name': 'branch_id', 'type': 'STRING', 'required': True}, {'name': 'account_type', 'type': 'STRING', 'required': True}, {'name': 'currency', 'type': 'STRING', 'required': True}, {'name': 'opened_at', 'type': 'TIMESTAMP', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'balance', 'type': 'DOUBLE', 'required': True}, {'name': 'credit_limit', 'type': 'DOUBLE', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'transactions': {'filename': 'transactions.csv', 'primary_key': ['transaction_id'], 'foreign_keys': [['account_id', 'accounts', 'account_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'transaction_id', 'type': 'STRING', 'required': True}, {'name': 'account_id', 'type': 'STRING', 'required': True}, {'name': 'event_time', 'type': 'TIMESTAMP', 'required': True}, {'name': 'transaction_type', 'type': 'STRING', 'required': True}, {'name': 'channel', 'type': 'STRING', 'required': True}, {'name': 'merchant_category', 'type': 'STRING', 'required': False}, {'name': 'currency', 'type': 'STRING', 'required': True}, {'name': 'amount', 'type': 'DOUBLE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}}
sources = {"accounts": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/banking/accounts/", "branches": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/banking/branches/", "customers": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/banking/customers/", "transactions": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/banking/transactions/"}
destinations = {"accounts": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/banking/accounts/", "branches": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/banking/branches/", "customers": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/banking/customers/", "transactions": f"oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/banking/transactions/"}
landing_tables = {"accounts": f"{participant_key}_banking_accounts", "branches": f"{participant_key}_banking_branches", "customers": f"{participant_key}_banking_customers", "transactions": f"{participant_key}_banking_transactions"}

spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_landing.{participant_key}_banking_branches (`participant_key` STRING, `source_row_id` STRING, `branch_id` STRING, `branch_type` STRING, `region` STRING, `opened_date` STRING, `status` STRING, `updated_at` STRING) USING CSV OPTIONS (header 'true') LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/banking/branches/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_landing.{participant_key}_banking_customers (`participant_key` STRING, `source_row_id` STRING, `customer_id` STRING, `customer_type` STRING, `segment` STRING, `region` STRING, `risk_band` STRING, `onboarding_date` STRING, `status` STRING, `updated_at` STRING) USING CSV OPTIONS (header 'true') LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/banking/customers/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_landing.{participant_key}_banking_accounts (`participant_key` STRING, `source_row_id` STRING, `account_id` STRING, `customer_id` STRING, `branch_id` STRING, `account_type` STRING, `currency` STRING, `opened_at` STRING, `status` STRING, `balance` STRING, `credit_limit` STRING, `updated_at` STRING) USING CSV OPTIONS (header 'true') LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/banking/accounts/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_landing.{participant_key}_banking_transactions (`participant_key` STRING, `source_row_id` STRING, `transaction_id` STRING, `account_id` STRING, `event_time` STRING, `transaction_type` STRING, `channel` STRING, `merchant_category` STRING, `currency` STRING, `amount` STRING, `status` STRING, `updated_at` STRING) USING CSV OPTIONS (header 'true') LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/banking/transactions/'""")

for dataset, spec in specs.items():
    frame = (spark.table(f"aidp_lab.oci_landing.{landing_tables[dataset]}")
        .withColumn("_source_file", F.input_file_name())
        .withColumn("_ingested_at", F.current_timestamp()))
    landing_count = frame.count()
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(destinations[dataset])
    bronze_count = spark.read.format("delta").load(destinations[dataset]).count()
    assert bronze_count == landing_count, f"Bronze count mismatch for {dataset}"
    print(f"Bronze {dataset}: {bronze_count} rows")

spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_bronze.{participant_key}_banking_branches (`participant_key` STRING, `source_row_id` STRING, `branch_id` STRING, `branch_type` STRING, `region` STRING, `opened_date` STRING, `status` STRING, `updated_at` STRING, `_source_file` STRING, `_ingested_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/banking/branches/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_bronze.{participant_key}_banking_customers (`participant_key` STRING, `source_row_id` STRING, `customer_id` STRING, `customer_type` STRING, `segment` STRING, `region` STRING, `risk_band` STRING, `onboarding_date` STRING, `status` STRING, `updated_at` STRING, `_source_file` STRING, `_ingested_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/banking/customers/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_bronze.{participant_key}_banking_accounts (`participant_key` STRING, `source_row_id` STRING, `account_id` STRING, `customer_id` STRING, `branch_id` STRING, `account_type` STRING, `currency` STRING, `opened_at` STRING, `status` STRING, `balance` STRING, `credit_limit` STRING, `updated_at` STRING, `_source_file` STRING, `_ingested_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/banking/accounts/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_bronze.{participant_key}_banking_transactions (`participant_key` STRING, `source_row_id` STRING, `transaction_id` STRING, `account_id` STRING, `event_time` STRING, `transaction_type` STRING, `channel` STRING, `merchant_category` STRING, `currency` STRING, `amount` STRING, `status` STRING, `updated_at` STRING, `_source_file` STRING, `_ingested_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/02_bronze/users/{participant_key}/banking/transactions/'""")


## Expected result

Four Landing CSV tables and four Bronze Delta tables are registered.

**Exercise:** rerun this notebook and confirm that counts do not increase. All writes use
participant-exclusive paths and overwrite mode, so a second run is idempotent.

**Common pitfall:** do not replace the participant paths with shared locations. That would mix
different students' data. As an extension, query the registered tables with `spark.sql`.
